# LeNet-5 from Scratch in PyTorch — MNIST Digit Classification

**Day 1 of 10 · Deep Learning Architectures Series**

In this notebook I implement the classic **LeNet-5** (LeCun et al., 1998) from scratch in PyTorch, train it on MNIST, and compare the original design (tanh + average pooling) with a modern variant (ReLU + max pooling).

**What's inside**
1. Background and architecture
2. Data pipeline
3. Model implementation + parameter count verification
4. Training and evaluation
5. Results: curves, confusion matrix, per-class accuracy, errors
6. Feature map visualization
7. Analysis, limitations, and next steps

> Paper: *Gradient-Based Learning Applied to Document Recognition* — LeCun, Bottou, Bengio, Haffner (1998)

> ⚙️ **Kaggle setup:** Settings → Accelerator → **GPU** (optional, CPU is fine for this model) and **Internet → On** (needed to download MNIST via torchvision).

## 1. Background and Architecture

LeNet-5 was one of the first successful convolutional neural networks, originally used to read handwritten digits on bank cheques. It introduced the core ideas still used today:

- **Local receptive fields and weight sharing:** the same 5×5 kernel slides across the entire image. Mathematically this is a 2D cross-correlation with a learnable FIR filter, which makes the network translation-equivariant with far fewer parameters than a fully connected network.
- **Subsampling (pooling):** reduces spatial resolution and adds small-shift invariance. From a signal-processing view, it is low-pass filtering followed by downsampling (decimation).
- **Feature hierarchy:** early layers detect strokes and edges, deeper layers combine them into digit parts and whole digits.

| Layer | Type | Output shape | Parameters |
|-------|------|--------------|-----------|
| Input | – | 1×32×32 | 0 |
| C1 | Conv 5×5, 6 filters | 6×28×28 | 156 |
| S2 | Pool 2×2 | 6×14×14 | 0 |
| C3 | Conv 5×5, 16 filters | 16×10×10 | 2,416 |
| S4 | Pool 2×2 | 16×5×5 | 0 |
| C5 | Conv 5×5, 120 filters | 120×1×1 | 48,120 |
| F6 | Fully connected | 84 | 10,164 |
| Output | Fully connected | 10 | 850 |
| **Total** | | | **61,706** |

**Output size of a conv layer:** $\lfloor (N - K + 2P)/S \rfloor + 1$. For C1: $(32 - 5)/1 + 1 = 28$.

**Parameters of a conv layer:** $C_{out} \times (C_{in} \times K \times K + 1)$. For C3: $16 \times (6 \times 25 + 1) = 2416$.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

## 2. Data Pipeline

MNIST images are 28×28, but LeNet-5 expects 32×32. In the original paper, the digit is centered in a 32×32 frame, so I **pad 2 pixels on each side**. Pixels are then normalized with the dataset mean and standard deviation.

In [ ]:
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3

tfm = transforms.Compose([
    transforms.Pad(2),                           # 28x28 -> 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean / std
])

train_ds = datasets.MNIST("./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=tfm)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512, shuffle=False)

print("Train samples:", len(train_ds), "| Test samples:", len(test_ds))

In [ ]:
# Quick look at the data
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, (img, label) in zip(axes.ravel(), [train_ds[i] for i in range(16)]):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(label, fontsize=10)
    ax.axis("off")
plt.suptitle("Sample MNIST training images (after padding to 32x32)")
plt.tight_layout()
plt.show()

## 3. Model Implementation

The class supports two variants through one flag:
- **Original:** `tanh` activation + `AvgPool2d` (as in the paper)
- **Modern:** `ReLU` activation + `MaxPool2d`

Note: the original paper's C3 layer used a sparse connection table between S2 and C3 maps. Like most modern implementations, I use full connectivity.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10, modern=False):
        super().__init__()
        act  = nn.ReLU if modern else nn.Tanh
        pool = nn.MaxPool2d if modern else nn.AvgPool2d

        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),   act(),   # C1: 6x28x28
            pool(kernel_size=2, stride=2),             # S2: 6x14x14
            nn.Conv2d(6, 16, kernel_size=5),  act(),   # C3: 16x10x10
            pool(kernel_size=2, stride=2),             # S4: 16x5x5
            nn.Conv2d(16, 120, kernel_size=5), act(),  # C5: 120x1x1
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(120, 84), act(),                 # F6
            nn.Linear(84, num_classes),                # Output
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = LeNet5()
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")
assert n_params == 61_706, "Parameter count does not match the paper's architecture!"

In [ ]:
# Layer-by-layer shape trace (sanity check against the table above)
x = torch.randn(1, 1, 32, 32)
for layer in model.features:
    x = layer(x)
    print(f"{layer.__class__.__name__:<10} -> {tuple(x.shape)}")
x = model.classifier(x)
print(f"{'Classifier':<10} -> {tuple(x.shape)}")

## 4. Training and Evaluation

- **Loss:** Cross-Entropy
- **Optimizer:** Adam (lr = 1e-3)
- **Epochs:** 10, batch size 128

I train both variants with the same seed and hyperparameters for a fair comparison.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            n          += x.size(0)
    return total_loss / n, correct / n


def train(modern):
    torch.manual_seed(SEED)
    model = LeNet5(modern=modern).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    hist = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_dl, criterion, optimizer)
        te_loss, te_acc = run_epoch(model, test_dl, criterion)
        for k, v in zip(hist, [tr_loss, te_loss, tr_acc, te_acc]):
            hist[k].append(v)
        print(f"Epoch {ep:02d}/{EPOCHS} | train loss {tr_loss:.4f}, acc {tr_acc:.4f} "
              f"| test loss {te_loss:.4f}, acc {te_acc:.4f}")
    hist["time"] = time.time() - t_start
    return model, hist

In [ ]:
print("=== Variant A: tanh + AvgPool (original) ===")
model_a, hist_a = train(modern=False)

In [ ]:
print("=== Variant B: ReLU + MaxPool (modern) ===")
model_b, hist_b = train(modern=True)

## 5. Results

In [ ]:
def plot_history(histories, names):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for h, name in zip(histories, names):
        ax[0].plot(h["train_loss"], "--", label=f"{name} train")
        ax[0].plot(h["test_loss"],  "-",  label=f"{name} test")
        ax[1].plot(h["train_acc"],  "--", label=f"{name} train")
        ax[1].plot(h["test_acc"],   "-",  label=f"{name} test")
    ax[0].set_title("Loss");     ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    ax[1].set_ylim(0.95, 1.0)
    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150)
    plt.show()

import os
os.makedirs("/kaggle/working", exist_ok=True)
plot_history([hist_a, hist_b], ["tanh+avg", "relu+max"])

In [ ]:
summary = pd.DataFrame({
    "Variant": ["tanh + AvgPool (original)", "ReLU + MaxPool (modern)"],
    "Final test acc": [hist_a["test_acc"][-1], hist_b["test_acc"][-1]],
    "Best test acc":  [max(hist_a["test_acc"]), max(hist_b["test_acc"])],
    "Final test loss":[hist_a["test_loss"][-1], hist_b["test_loss"][-1]],
    "Epochs to reach 98%": [
        next((i+1 for i, a in enumerate(hist_a["test_acc"]) if a >= 0.98), None),
        next((i+1 for i, a in enumerate(hist_b["test_acc"]) if a >= 0.98), None)],
    "Train time (s)": [round(hist_a["time"], 1), round(hist_b["time"], 1)],
})
summary

In [ ]:
@torch.no_grad()
def predict_all(model):
    model.eval()
    preds, labels, imgs = [], [], []
    for x, y in test_dl:
        preds.append(model(x.to(DEVICE)).argmax(1).cpu())
        labels.append(y); imgs.append(x)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy(), torch.cat(imgs)

best_model = model_a if hist_a["test_acc"][-1] >= hist_b["test_acc"][-1] else model_b
best_name  = "tanh + AvgPool" if best_model is model_a else "ReLU + MaxPool"
preds, labels, imgs = predict_all(best_model)

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix — {best_name}")
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(labels, preds, digits=4))

In [ ]:
# Most confused digit pairs
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
top = np.dstack(np.unravel_index(np.argsort(cm_off.ravel())[::-1][:5], cm.shape))[0]
print("Top-5 confusions (true -> predicted : count)")
for t, p in top:
    print(f"  {t} -> {p} : {cm_off[t, p]}")

# Misclassified examples
wrong = np.where(preds != labels)[0][:16]
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(imgs[i].squeeze(), cmap="gray")
    ax.set_title(f"T:{labels[i]} P:{preds[i]}", fontsize=9, color="red")
    ax.axis("off")
plt.suptitle("Misclassified test images (T = true, P = predicted)")
plt.tight_layout()
plt.savefig("/kaggle/working/misclassified.png", dpi=150)
plt.show()

In [ ]:
import seaborn as sns

# ---------- 1) Overfitting analysis ----------
epochs = np.arange(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for h, name, c in [(hist_a, "tanh+avg", "tab:blue"), (hist_b, "relu+max", "tab:red")]:
    acc_gap  = np.array(h["train_acc"]) - np.array(h["test_acc"])     # > 0 means overfitting
    loss_gap = np.array(h["test_loss"]) - np.array(h["train_loss"])   # > 0 means overfitting
    ax[0].plot(epochs, acc_gap * 100, marker="o", color=c, label=name)
    ax[1].plot(epochs, loss_gap,      marker="o", color=c, label=name)

for a, title, ylab in zip(ax,
        ["Accuracy gap (train - test)", "Loss gap (test - train)"],
        ["percentage points", "loss"]):
    a.axhline(0, color="gray", lw=0.8, ls="--")
    a.set_title(title); a.set_xlabel("epoch"); a.set_ylabel(ylab)
    a.legend(); a.grid(alpha=0.3)

plt.suptitle("Overfitting analysis: generalization gap per epoch")
plt.tight_layout()
plt.savefig("/kaggle/working/overfitting_gap.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- 2) Confusion matrix heatmaps ----------
cm_err = cm.copy()
np.fill_diagonal(cm_err, 0)   # hide correct predictions to highlight errors

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=range(10), yticklabels=range(10))
ax[0].set_title(f"Confusion matrix (counts) - {best_name}")

sns.heatmap(cm_err, annot=True, fmt="d", cmap="Reds", cbar=False, ax=ax[1],
            xticklabels=range(10), yticklabels=range(10))
ax[1].set_title("Errors only (diagonal removed)")

for a in ax:
    a.set_xlabel("Predicted"); a.set_ylabel("True")

plt.tight_layout()
plt.savefig("/kaggle/working/confusion_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. What Did the Network Learn? — Feature Maps

Below: the 6 learned 5×5 filters of C1, and the feature maps they produce for one test digit. Early filters typically act as oriented stroke/edge detectors.

In [ ]:
conv1 = best_model.features[0]
filters = conv1.weight.detach().cpu()          # (6, 1, 5, 5)

# --- C1 filters ---
fig, axes = plt.subplots(1, 6, figsize=(12, 2.3))
for i, ax in enumerate(axes):
    ax.imshow(filters[i, 0], cmap="gray")
    ax.set_title(f"filter {i}"); ax.axis("off")
plt.suptitle("Learned C1 filters (5x5)")
plt.savefig("/kaggle/working/c1_filters.png", dpi=150, bbox_inches="tight")
plt.show()

# --- C1 feature maps ---
sample = imgs[0:1].to(DEVICE)
with torch.no_grad():
    fmap = best_model.features[1](conv1(sample)).cpu()
fig, axes = plt.subplots(1, 7, figsize=(14, 2.4))
axes[0].imshow(imgs[0].squeeze(), cmap="gray")
axes[0].set_title(f"input (label {labels[0]})"); axes[0].axis("off")
for i in range(6):
    axes[i+1].imshow(fmap[0, i], cmap="viridis")
    axes[i+1].set_title(f"map {i}"); axes[i+1].axis("off")
plt.suptitle("C1 feature maps for one test image")
plt.savefig("/kaggle/working/c1_feature_maps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save trained weights
torch.save(model_a.state_dict(), "/kaggle/working/lenet5_tanh_avgpool.pth")
torch.save(model_b.state_dict(), "/kaggle/working/lenet5_relu_maxpool.pth")
print("Saved model weights to /kaggle/working/")

## 7. Analysis

**Variant comparison.** The two variants end up very close: 99.02% (original) vs 98.88% (modern), a gap of 14 images out of 10,000. The modern variant reached 98% accuracy one epoch earlier (epoch 2 vs 3), which is consistent with ReLU avoiding the saturation that slows tanh early in training. However, the original design finished with slightly higher accuracy and lower test loss. Since this is a single run per variant with one seed, the final gap is within what I would expect from run-to-run noise, so I cannot claim that either variant is better on MNIST. MNIST is also too easy to separate such small architectural differences. Training time was nearly identical (~180 s), and for a model this small the cost is dominated by data loading rather than compute.

**Per-class behavior.** Performance is uniformly high (all F1 > 0.98). The weakest class is **digit 9** (recall 0.9713, F1 0.9829): about 3% of true 9s are predicted as something else. The lowest precisions belong to **7** (0.9808) and **4** (0.9818), which suggests that wrongly classified 9s tend to land in those classes, which makes sense since 4, 7 and 9 share similar strokes (a closed or semi-closed top with a vertical stem). Digit **3** has perfect recall (1.0000) but lower precision (0.9883), meaning the model slightly over-predicts 3 and some other digits get mistaken for it. The confusion matrix confirms this: the two most frequent errors are **9 -> 4 (12 cases)** and **9 -> 7 (8 cases)**, followed by 7 -> 2, 9 -> 8 and 6 -> 4 (4 cases each). Looking at the misclassified images, many of the errors are genuinely ambiguous or unusual handwriting (a heavily slanted or rotated 6 that looks like a 0, a 4 with a closed top that resembles a 9, a 7 with a crossbar that resembles a 2), where even a human reader could hesitate. This suggests the model is close to the ceiling of what a small CNN without augmentation can achieve on MNIST, and that the remaining errors are mostly hard samples, not systematic bugs.

**Learned features.** The six 5x5 filters in C1 are coarse and not perfectly clean (5x5 is a very small support), but they behave as oriented stroke/edge detectors. For example, filter 1 has dark pixels on top and bright pixels at the bottom (a horizontal-edge detector), while filter 4 has a dark vertical band on the right (a vertical-edge detector). In the feature maps for a test "7", maps 0, 1 and 5 respond strongly to the horizontal top bar, maps 2 and 4 respond to the boundaries of the diagonal stroke, and map 3 produces a double-edge outline of the stroke. Some maps are polarity-inverted versions of each other (bright vs. dark stroke), which is expected since filters can learn positive or negative weights. Together, C1 decomposes the digit into horizontal and diagonal stroke components that deeper layers combine.

**Generalization.** Train accuracy keeps climbing to about 99.5% while test accuracy plateaus around 98.7-99.0% after epoch ~3-4. Likewise, train loss keeps decreasing (to ~0.015) while test loss flattens around 0.035-0.045. The gap of roughly 0.5 percentage points indicates mild overfitting that starts after the first few epochs, not severe overfitting. Test curves are also noisy from epoch to epoch (swings of about 0.2-0.3 pp, especially for ReLU + MaxPool), which is another reason not to over-interpret the small final difference between the two variants. Early stopping, augmentation or regularization would be natural next steps.

### Limitations
- Single run per variant, so no confidence intervals; differences below ~0.2% are not conclusive.
- MNIST is nearly saturated, and a harder dataset would better separate the designs.
- No augmentation, regularization, or LR scheduling.
- Fully connected C3 instead of the paper's sparse connection table.

### What I would improve
- Run 5 seeds per variant and report mean +/- std
- Add augmentation (small shifts/rotations) and test robustness to shifted digits
- Add BatchNorm / Dropout and an LR scheduler (e.g. OneCycle)
- Repeat on Fashion-MNIST and CIFAR-10 to compare the variants where the difference matters

### What I learned
- How convolution, pooling and weight sharing reduce parameters while building a feature hierarchy
- How to verify an implementation (parameter count, shape tracing) against a paper
- Why single-run differences on an easy benchmark should be interpreted carefully


### Next in this series
**Day 2:** AlexNet/VGG-style network on CIFAR-10 with augmentation, BatchNorm, and Dropout.

### References
- LeCun, Y., Bottou, L., Bengio, Y., Haffner, P. (1998). *Gradient-Based Learning Applied to Document Recognition.* Proceedings of the IEEE.